In [ ]:
!pip install kaggle opencv-python-headless matplotlib seaborn pillow -q

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from collections import Counter



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

zip_path = '/content/drive/MyDrive/datasets_v2.zip'
extract_path = '/content/datasets/'

print('Unzipping...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print('Done!')

BASE = '/content/datasets/datasets'
created_dir = os.path.join(BASE, 'created_dataset')
levels = ['Level_0', 'Level_1', 'Level_2']

In [ ]:
import os, cv2, numpy as np, pandas as pd

BASE = '/content/datasets/datasets'

records = []
for level in ['Level_0', 'Level_1', 'Level_2']:
    folder = os.path.join(BASE, 'created_dataset', level)
    for imgname in os.listdir(folder):
        if not imgname.lower().endswith(('.jpg','.png','.jpeg')):
            continue
        img = cv2.imread(os.path.join(folder, imgname))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        records.append({
            'path': os.path.join(folder, imgname),
            'brightness': np.mean(gray),
            'redness': np.mean(img[:,:,0]),
            'saturation': np.mean(hsv[:,:,1]),
            'gradient': np.mean(np.abs(np.gradient(gray.astype(float)))),
            'contrast': gray.std(),
            'edges': cv2.Canny(gray, 100, 200).mean(),
            'label': level
        })

df = pd.DataFrame(records)
df['label_num'] = df['label'].map({'Level_0':0,'Level_1':1,'Level_2':2})
df['brightness_redness_ratio'] = df['brightness'] / (df['redness'] + 1e-8)
df['contrast_brightness_ratio'] = df['contrast'] / (df['brightness'] + 1e-8)
df['log_edges'] = np.log1p(df['edges'])
df['log_contrast'] = np.log1p(df['contrast'])

print(f'✅ df ready: {df.shape}')
print(df['label'].value_counts())

In [ ]:
import os
import shutil
import random

BASE = '/content/datasets/datasets'

# create unified folder structure
for split in ['train', 'val', 'test']:
    for level in ['Level_0', 'Level_1', 'Level_2']:
        os.makedirs(f'{BASE}/unified/{split}/{level}', exist_ok=True)

# collect all images from created_dataset
all_images = {'Level_0': [], 'Level_1': [], 'Level_2': []}

for level in ['Level_0', 'Level_1', 'Level_2']:
    folder = f'{BASE}/created_dataset/{level}'
    imgs = [f for f in os.listdir(folder)
            if f.endswith(('.jpg', '.png', '.jpeg'))]
    all_images[level] = [(os.path.join(folder, f), f) for f in imgs]
    print(f'{level}: {len(imgs)} images')

# split 70/15/15 per class
random.seed(42)
total_copied = 0

for level, img_list in all_images.items():
    random.shuffle(img_list)
    n = len(img_list)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    splits = {
        'train': img_list[:train_end],
        'val':   img_list[train_end:val_end],
        'test':  img_list[val_end:]
    }

    for split, imgs in splits.items():
        for src_path, fname in imgs:
            dst = f'{BASE}/unified/{split}/{level}/{fname}'
            shutil.copy2(src_path, dst)
            total_copied += 1

print(f'\nTotal copied: {total_copied}')
print('\n=== Unified dataset summary ===')
for split in ['train', 'val', 'test']:
    split_total = 0
    for level in ['Level_0', 'Level_1', 'Level_2']:
        count = len(os.listdir(f'{BASE}/unified/{split}/{level}'))
        print(f'  {split}/{level}: {count}')
        split_total += count
    print(f'  {split} total: {split_total}\n')

In [ ]:
import matplotlib.pyplot as plt
import os

BASE = '/content/datasets/datasets'

counts = {}
for level in ['Level_0', 'Level_1', 'Level_2']:
    folder = os.path.join(BASE, 'created_dataset', level)
    counts[level] = len([f for f in os.listdir(folder)
                         if f.endswith(('.jpg','.png','.jpeg'))])

labels = ['Level 0\n(mild)', 'Level 1\n(moderate)', 'Level 2\n(severe)']
colors = ['#4CAF50', '#FF9800', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, counts.values(), color=colors)
axes[0].set_title('Image count per severity level\n(4,621 total images)', fontsize=13)
axes[0].set_ylabel('Number of images')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values(), labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class distribution %', fontsize=13)

plt.suptitle('Dataset Overview — Acne Severity Classification', fontsize=14)
plt.tight_layout()
plt.show()

print('Class counts:')
for k, v in counts.items():
    print(f'  {k}: {v} images ({v/sum(counts.values())*100:.1f}%)')

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

BASE = '/content/datasets/datasets'
label_names = {
    'Level_0': 'Level 0 - Mild',
    'Level_1': 'Level 1 - Moderate',
    'Level_2': 'Level 2 - Severe'
}

fig, axes = plt.subplots(3, 5, figsize=(20, 13))

for row, level in enumerate(['Level_0', 'Level_1', 'Level_2']):
    folder = os.path.join(BASE, 'created_dataset', level)
    all_imgs = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    samples = random.sample(all_imgs, 5)

    for col, img_name in enumerate(samples):
        img = Image.open(os.path.join(folder, img_name))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_title(f"{label_names[level]}\n{img_name}", fontsize=9)

plt.suptitle('Sample images by acne severity level', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

BASE = '/content/datasets/datasets'
widths, heights, level_tags = [], [], []

for level in ['Level_0', 'Level_1', 'Level_2']:
    folder = os.path.join(BASE, 'created_dataset', level)
    for img_name in os.listdir(folder):
        if not img_name.endswith(('.jpg','.png','.jpeg')):
            continue
        img = cv2.imread(os.path.join(folder, img_name))
        if img is None:
            continue
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)
        level_tags.append(level)

import pandas as pd
size_df = pd.DataFrame({'width': widths, 'height': heights, 'level': level_tags})

print('Width stats:')
print(size_df['width'].describe().round(1))
print('\nHeight stats:')
print(size_df['height'].describe().round(1))
print(f'\nUnique sizes: {size_df[["width","height"]].drop_duplicates().shape[0]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Level_0': '#4CAF50', 'Level_1': '#FF9800', 'Level_2': '#F44336'}
for level, color in colors.items():
    subset = size_df[size_df['level'] == level]
    axes[0].scatter(subset['width'], subset['height'],
                   alpha=0.3, label=level, color=color, s=10)
axes[0].set_title('Image size distribution')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].legend()

axes[1].hist(size_df['height'], bins=30, color='steelblue', edgecolor='black')
axes[1].set_title('Height distribution')
axes[1].set_xlabel('Height (px)')
axes[1].set_ylabel('Count')

plt.suptitle('Image size analysis across all 4,621 images', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# using df from your existing feature extraction cell
features = ['brightness', 'redness', 'saturation', 'contrast', 'edges']
titles = ['Brightness', 'Redness', 'Saturation', 'Contrast', 'Edge density']
colors = {'Level_0': '#4CAF50', 'Level_1': '#FF9800', 'Level_2': '#F44336'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (feature, title) in enumerate(zip(features, titles)):
    for level, color in colors.items():
        subset = df[df['label'] == level][feature]
        axes[idx].hist(subset, alpha=0.6, label=level,
                      color=color, bins=30)
    axes[idx].set_title(title, fontsize=12)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Count')
    axes[idx].legend()

axes[5].axis('off')
plt.suptitle('Feature distributions by severity level — 4,621 images', fontsize=14)
plt.tight_layout()
plt.show()

print('Mean feature values by severity level:')
print(df.groupby('label')[features].mean().round(2))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# map labels to numbers for correlation
df['label_num'] = df['label'].map({'Level_0': 0, 'Level_1': 1, 'Level_2': 2})

features = ['brightness', 'redness', 'saturation', 'gradient', 'contrast', 'edges', 'label_num']

plt.figure(figsize=(10, 8))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True,
            xticklabels=['Brightness','Redness','Saturation','Gradient','Contrast','Edges','Label'],
            yticklabels=['Brightness','Redness','Saturation','Gradient','Contrast','Edges','Label'])
plt.title('Feature correlation matrix\n(Label = severity level)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nCorrelation with severity label:')
print(df[features].corr()['label_num'].drop('label_num').sort_values(ascending=False).round(3))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

palette = {'Level_0': '#4CAF50', 'Level_1': '#FF9800', 'Level_2': '#F44336'}

# pairplot of top features
plot_df = df[['brightness', 'contrast', 'redness', 'log_edges', 'label']].sample(500, random_state=42)

g = sns.pairplot(plot_df, hue='label', palette=palette,
                 plot_kws={'alpha': 0.4, 's': 20},
                 diag_kind='kde')
g.fig.suptitle('Pairplot of key features — 500 sample images', y=1.02, fontsize=13)
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

features = ['brightness', 'redness', 'saturation', 'contrast',
            'edges', 'gradient', 'brightness_redness_ratio',
            'contrast_brightness_ratio', 'log_edges']

X = df[features].values
y = df['label_num'].values

# standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print('Explained variance ratio per component:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.3f} ({var*100:.1f}%)')
print(f'  Total: {pca.explained_variance_ratio_.sum():.3f}')

print('\nFeature loadings on PC1 (most important component):')
loadings = pd.DataFrame(pca.components_.T,
                        index=features,
                        columns=['PC1','PC2','PC3'])
print(loadings['PC1'].sort_values(ascending=False).round(3))

# plot PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = {0: '#4CAF50', 1: '#FF9800', 2: '#F44336'}
label_map = {0: 'Mild', 1: 'Moderate', 2: 'Severe'}

for label_val in [0, 1, 2]:
    mask = y == label_val
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_map[label_val],
                   label=label_map[label_val],
                   alpha=0.4, s=15)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].set_title('PCA — PC1 vs PC2')
axes[0].legend()

for label_val in [0, 1, 2]:
    mask = y == label_val
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 2],
                   c=colors_map[label_val],
                   label=label_map[label_val],
                   alpha=0.4, s=15)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}% variance)')
axes[1].set_title('PCA — PC1 vs PC3')
axes[1].legend()

plt.suptitle('PCA projection — can handcrafted features separate severity classes?', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

features = ['brightness', 'redness', 'saturation', 'contrast', 'edges']

# z-score outlier detection
z_scores = np.abs(stats.zscore(df[features]))
outlier_mask = (z_scores > 3).any(axis=1)

print(f'Total outliers (z-score > 3): {outlier_mask.sum()} images ({outlier_mask.mean()*100:.1f}%)')
print('\nOutliers per severity level:')
print(df[outlier_mask]['label'].value_counts())

print('\nOutlier feature breakdown:')
for feat in features:
    n = (np.abs(stats.zscore(df[feat])) > 3).sum()
    print(f'  {feat}: {n} outliers')

# visualize outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_out = np.where(outlier_mask, 'red', 'steelblue')
axes[0].scatter(df['brightness'], df['contrast'],
               c=colors_out, alpha=0.3, s=10)
axes[0].set_xlabel('Brightness')
axes[0].set_ylabel('Contrast')
axes[0].set_title('Outliers in brightness vs contrast space\n(red = outlier)')

# show what outlier images look like
import cv2
import os

BASE = '/content/datasets/datasets'
outlier_df = df[outlier_mask].head(6)

fig2, axes2 = plt.subplots(2, 3, figsize=(14, 8))
for idx, (_, row) in enumerate(outlier_df.iterrows()):
    ax = axes2[idx//3, idx%3]
    img = cv2.imread(row['path'] if 'path' in df.columns
                    else os.path.join(BASE, 'created_dataset',
                    row['label'], os.listdir(
                    os.path.join(BASE, 'created_dataset', row['label']))[idx]))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        ax.imshow(img)
    ax.set_title(f"{row['label']}\nbright={row['brightness']:.0f}, contrast={row['contrast']:.0f}", fontsize=8)
    ax.axis('off')

plt.suptitle('Sample outlier images', fontsize=13)
plt.tight_layout()
plt.show()

## Sampling Considerations

Our dataset is not a random sample of all acne patients. Key biases:

1. Geographic bias — ACNE04 and IGA datasets are primarily East Asian
   subjects, which may not generalize globally
2. Severity bias — severe cases (16.8%) are underrepresented vs real
   population distributions
3. Image quality bias — clinical photos have controlled lighting unlike
   real-world selfies
4. Label bias — different grading scales (Hayashi vs IGA) merged together
   may introduce inconsistency

These biases mean our model may perform differently on real-world data
than our test set accuracy suggests.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

feature_cols = ['brightness', 'redness', 'saturation', 'contrast',
                'edges', 'gradient', 'brightness_redness_ratio',
                'contrast_brightness_ratio', 'log_edges']

X = df[feature_cols].values
y = df['label_num'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)

print(f'Baseline Logistic Regression (4,621 images): {acc:.2%}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Mild', 'Moderate', 'Severe']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Mild','Moderate','Severe'],
            yticklabels=['Mild','Moderate','Severe'])
plt.title(f'Baseline Confusion Matrix — {acc:.2%} accuracy\n(4,621 images, 9 engineered features)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BASE = '/content/datasets/datasets'
IMG_SIZE = 224
BATCH_SIZE = 32

# augmentation for training only
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{BASE}/unified/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    f'{BASE}/unified/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    f'{BASE}/unified/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')
print(f'Classes: {train_gen.class_indices}')

# class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_gen.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

# build model
base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model3 = tf.keras.Model(inputs, outputs)
model3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, verbose=1)
]

print('\nTraining CNN on 4,621 images with augmentation...')
history3 = model3.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# evaluate on held out test set
y_pred_probs = model3.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

acc = np.mean(y_pred == y_true)

print('=== Model Comparison ===')
print(f'Baseline LR (999 images):       53.50%')
print(f'CNN v1 (999 images):            64.50%')
print(f'CNN v2 (4,621 images):          {acc:.2%}')

print('\nClassification Report:')
print(classification_report(y_true, y_pred,
      target_names=['Mild', 'Moderate', 'Severe']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Mild','Moderate','Severe'],
            yticklabels=['Mild','Moderate','Severe'])
plt.title(f'CNN v2 Confusion Matrix\nAccuracy: {acc:.2%} | Test set: 695 images')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history3.history['accuracy'], label='Train', color='blue')
axes[0].plot(history3.history['val_accuracy'], label='Validation', color='orange')
axes[0].axhline(y=0.535, color='red', linestyle='--', label='LR Baseline (53.5%)')
axes[0].axhline(y=0.645, color='purple', linestyle='--', label='CNN v1 (64.5%)')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history3.history['loss'], label='Train', color='blue')
axes[1].plot(history3.history['val_loss'], label='Validation', color='orange')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.suptitle('CNN v2 Training History — 4,621 images + augmentation', fontsize=13)
plt.tight_layout()
plt.show()

# save model
model3.save('/content/drive/MyDrive/acne_model_v2.keras')
print('\nModel saved to Drive as acne_model_v2.keras')

In [ ]:
# ============================================================
# CNN v3 — EfficientNetB0 Transfer Learning
# Different backbone from MobileNetV2; compound scaling makes
# it more accurate at similar parameter count
# ============================================================
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

BASE = '/content/datasets/datasets'
IMG_SIZE = 224
BATCH_SIZE = 32

# EfficientNet expects pixel values in [0, 255] — it has its own
# internal rescaling via the include_preprocessing flag (True by default)
# So we do NOT divide by 255 here
train_datagen_eff = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)
val_datagen_eff   = ImageDataGenerator()
test_datagen_eff  = ImageDataGenerator()

train_gen_eff = train_datagen_eff.flow_from_directory(
    f'{BASE}/unified/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
val_gen_eff = val_datagen_eff.flow_from_directory(
    f'{BASE}/unified/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
test_gen_eff = test_datagen_eff.flow_from_directory(
    f'{BASE}/unified/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

labels_eff = train_gen_eff.classes
class_weights_eff = compute_class_weight('balanced', classes=np.unique(labels_eff), y=labels_eff)
class_weight_dict_eff = dict(enumerate(class_weights_eff))

# Build EfficientNetB0 model
base_eff = EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_eff.trainable = False  # frozen backbone — same fair comparison as model3

inputs_eff = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_eff(inputs_eff, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)          # extra stability vs model3
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs_eff = layers.Dense(3, activation='softmax')(x)

model_eff = tf.keras.Model(inputs_eff, outputs_eff)
model_eff.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_eff.summary()

callbacks_eff = [
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

print('\nTraining CNN v3 — EfficientNetB0...')
history_eff = model_eff.fit(
    train_gen_eff,
    epochs=30,
    validation_data=val_gen_eff,
    class_weight=class_weight_dict_eff,
    callbacks=callbacks_eff,
    verbose=1
)

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred_eff = np.argmax(model_eff.predict(test_gen_eff), axis=1)
y_true_eff = test_gen_eff.classes
acc_eff = np.mean(y_pred_eff == y_true_eff)

print(f'\n=== CNN v3 (EfficientNetB0) Test Accuracy: {acc_eff:.2%} ===')
print(classification_report(y_true_eff, y_pred_eff,
      target_names=['Mild', 'Moderate', 'Severe']))

cm_eff = confusion_matrix(y_true_eff, y_pred_eff)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cm_eff, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Mild','Moderate','Severe'],
            yticklabels=['Mild','Moderate','Severe'], ax=axes[0])
axes[0].set_title(f'EfficientNetB0 Confusion Matrix\nAccuracy: {acc_eff:.2%}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

axes[1].plot(history_eff.history['accuracy'],     label='Train', color='blue')
axes[1].plot(history_eff.history['val_accuracy'], label='Val',   color='orange')
axes[1].axhline(y=0.535, color='red',    linestyle='--', label='LR Baseline (53.5%)')
axes[1].axhline(y=0.645, color='purple', linestyle='--', label='CNN v1 (64.5%)')
axes[1].set_title('EfficientNetB0 Training Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('CNN v3 — EfficientNetB0 Results', fontsize=13)
plt.tight_layout()
plt.show()

model_eff.save('/content/drive/MyDrive/acne_model_efficientnet.keras')
print('Saved to Drive.')

In [ ]:
# ============================================================
# CNN v4 revised — Fine-tuned MobileNetV2 targeting 70%+
# ============================================================
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np, seaborn as sns, matplotlib.pyplot as plt

BASE = '/content/datasets/datasets'
IMG_SIZE = 224
BATCH_SIZE = 32

# more aggressive augmentation than before
train_datagen_ft = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    zoom_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    fill_mode='nearest'
)
val_datagen_ft  = ImageDataGenerator(rescale=1./255)
test_datagen_ft = ImageDataGenerator(rescale=1./255)

train_gen_ft = train_datagen_ft.flow_from_directory(
    f'{BASE}/unified/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen_ft = val_datagen_ft.flow_from_directory(
    f'{BASE}/unified/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen_ft = test_datagen_ft.flow_from_directory(
    f'{BASE}/unified/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

labels_ft = train_gen_ft.classes
cw = compute_class_weight('balanced', classes=np.unique(labels_ft), y=labels_ft)
cw_dict = dict(enumerate(cw))

# ── Build model ────────────────────────────────────────────────────
base_ft = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                      include_top=False, weights='imagenet')
base_ft.trainable = False

inputs_ft = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_ft(inputs_ft, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)        # helps stabilize fine-tuning
x = layers.Dense(512, activation='relu')(x)   # bigger head than before
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs_ft = layers.Dense(3, activation='softmax')(x)

model_ft = tf.keras.Model(inputs_ft, outputs_ft)

# ── Phase 1: train head only, higher LR, no label smoothing ───────
model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',       # no smoothing in phase 1
    metrics=['accuracy']
)

print('Phase 1 — training head only...')
history_ft1 = model_ft.fit(
    train_gen_ft, epochs=20,
    validation_data=val_gen_ft,
    class_weight=cw_dict,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-5, verbose=1)
    ],
    verbose=1
)

# ── Phase 2: unfreeze top layers, continue from Phase 1 weights ────
print(f'\nTotal backbone layers: {len(base_ft.layers)}')
UNFREEZE_FROM = 100    # ~54 layers — conservative to preserve what Phase 1 learned

base_ft.trainable = True
for layer in base_ft.layers[:UNFREEZE_FROM]:
    layer.trainable = False

# keep BatchNorm frozen during fine-tuning — prevents distribution shift
for layer in base_ft.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# slightly higher LR than before (1e-5 vs 5e-6) so it actually moves
# label smoothing only in phase 2 to handle Moderate ambiguity
model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

print(f'Phase 2 — fine-tuning top {len(base_ft.layers) - UNFREEZE_FROM} backbone layers...')
history_ft2 = model_ft.fit(
    train_gen_ft, epochs=25,
    validation_data=val_gen_ft,
    class_weight=cw_dict,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=4, min_lr=1e-8, verbose=1),
        ModelCheckpoint('/content/drive/MyDrive/acne_best_ft.keras',
                        monitor='val_accuracy', save_best_only=True, verbose=1)
    ],
    verbose=1
)

# ── Evaluate ───────────────────────────────────────────────────────
y_pred_ft = np.argmax(model_ft.predict(test_gen_ft), axis=1)
y_true_ft = test_gen_ft.classes
acc_ft = np.mean(y_pred_ft == y_true_ft)

print(f'\n=== CNN v4 Revised Test Accuracy: {acc_ft:.2%} ===')
print(classification_report(y_true_ft, y_pred_ft,
      target_names=['Mild', 'Moderate', 'Severe']))

acc_all = history_ft1.history['accuracy'] + history_ft2.history['accuracy']
val_all  = history_ft1.history['val_accuracy'] + history_ft2.history['val_accuracy']
phase2_start = len(history_ft1.history['accuracy'])

cm_ft = confusion_matrix(y_true_ft, y_pred_ft)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cm_ft, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Mild','Moderate','Severe'],
            yticklabels=['Mild','Moderate','Severe'], ax=axes[0])
axes[0].set_title(f'Fine-tuned MobileNetV2 (Revised)\nAccuracy: {acc_ft:.2%}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

axes[1].plot(acc_all, label='Train', color='blue')
axes[1].plot(val_all, label='Val',   color='orange')
axes[1].axvline(x=phase2_start, color='gray', linestyle=':', label='Fine-tune starts')
axes[1].axhline(y=0.535,  color='red',    linestyle='--', label='LR Baseline (53.5%)')
axes[1].axhline(y=0.645,  color='purple', linestyle='--', label='CNN v1 (64.5%)')
axes[1].axhline(y=0.6374, color='green',  linestyle='--', label='EfficientNetB0 (63.74%)')
axes[1].axhline(y=0.70,   color='black',  linestyle='--', label='Target (70%)', linewidth=2)
axes[1].set_title('Fine-tuned MobileNetV2 Training Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('CNN v4 Revised — Fine-tuned MobileNetV2', fontsize=13)
plt.tight_layout()
plt.show()

model_ft.save('/content/drive/MyDrive/acne_model_finetuned_v2.keras')
print('Saved.')

In [ ]:
from google.colab import files
import numpy as np
from tensorflow.keras.preprocessing import image
import io

def upload_and_predict():
    # 1. Create the upload button
    uploaded = files.upload()

    for filename in uploaded.keys():
        # 2. Process the uploaded file
        # We use io.BytesIO so we can read the image directly from memory
        img = image.load_img(io.BytesIO(uploaded[filename]), target_size=(224, 224))

        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array /= 255.0

        # 3. Predict (Make sure you've loaded model_ft first!)
        try:
            probs = model_ft.predict(img_array)
            pred_idx = np.argmax(probs, axis=1)[0]

            labels = {0: 'Mild', 1: 'Moderate', 2: 'Severe'}

            print(f"\n--- Results for: {filename} ---")
            print(f"Prediction: {labels[pred_idx]}")
            print(f"Confidence: {probs[0][pred_idx]:.1%}")
            print(f"Details -> Mild: {probs[0][0]:.1%} | Moderate: {probs[0][1]:.1%} | Severe: {probs[0][2]:.1%}")
        except NameError:
            print("Error: 'model_ft' is not defined. Please run your training or loading code first!")

# Start the prompt
upload_and_predict()